# Scatter-Gather on WANDS

This notebook recreates the `scatter_gather_wands` strategy from `configs/cheat-at-search/scatter_gather_wands.yml` using cheat-at-search utilities. It selects categories, scatters across category-filtered tools, and gathers a final ranked list.

In [ ]:
# Install cheat-at-search at the repo commit used for this experiment
!pip -q install git+https://github.com/softwaredoug/cheat-at-search.git@87d569c368059448f9ee036e10a7afec622c71c5

In [ ]:
# Mount data directory (Colab-friendly, falls back to local path)
from cheat_at_search.data_dir import mount
try:
    mount(use_gdrive=True)
except ImportError:
    from pathlib import Path
    manual_path = str(Path.home() / ".search-experiments" / "cheat-at-search")
    mount(use_gdrive=False, manual_path=manual_path)

## Load dataset and OpenAI client

This pulls the WANDS corpus and judgments and loads an OpenAI key for agentic steps.

In [ ]:
from cheat_at_search.data_dir import key_for_provider
from openai import OpenAI
from cheat_at_search.wands_data import corpus, judgments

OPENAI_KEY = key_for_provider("openai")
openai = OpenAI(api_key=OPENAI_KEY)

corpus.head(3)

## Build tools used by scatter/gather

We implement a subset of the repo tools: top categories, BM25 with optional category filter, and embedding search with a category filter.

In [ ]:
import numpy as np
from searcharray import SearchArray
from cheat_at_search.tokenizers import snowball_tokenizer
from cheat_at_search.embeddings import DEFAULT_MODEL_NAME, load_or_create_embeddings, load_model

CATEGORY_COL = "category"
TOP_CATEGORIES = [
    "Furniture",
    "Home Improvement",
    "Decor & Pillows",
    "Outdoor",
    "Storage & Organization",
    "Lighting",
    "Rugs",
    "Bed & Bath",
    "Kitchen & Tabletop",
    "Baby & Kids",
    "School Furniture and Supplies",
    "Appliances",
    "Holiday Decor",
    "Commercial Business Furniture",
    "Pet",
    "Contractor",
    "Sale",
    "Foodservice",
    "Shop Product Type",
    "Browse By Brand",
]

def _category_index(df, column=CATEGORY_COL):
    values = df[column].fillna("").astype(str).tolist()
    index = {}
    for idx, value in enumerate(values):
        if not value:
            continue
        index.setdefault(value, []).append(idx)
    return {key: np.asarray(indices, dtype=int) for key, indices in index.items()}

def top_categories(top_k=5, column=CATEGORY_COL):
    if column not in corpus.columns:
        raise ValueError(f"Missing {column} column in corpus")
    values = set(corpus[column].dropna().astype(str).tolist())
    ordered = [cat for cat in TOP_CATEGORIES if cat in values]
    missing = sorted(values - set(TOP_CATEGORIES))
    categories = ordered + missing
    trimmed = categories[:top_k]
    remaining = max(len(categories) - top_k, 0)
    if remaining:
        trimmed.append(f"truncated ({remaining} other categories)")
    return trimmed

def bm25_wands(keywords, product_categories=None, top_k=10, column=CATEGORY_COL):
    if top_k > 100:
        raise ValueError("top_k must be <= 100")
    index = _category_index(corpus, column=column)
    if product_categories is None:
        working = corpus
    else:
        if isinstance(product_categories, str):
            product_categories = [product_categories]
        indices = []
        for cat in product_categories:
            if cat in index:
                indices.append(index[cat])
        if not indices:
            return []
        working = corpus.iloc[np.unique(np.concatenate(indices))]
    scores = np.zeros(len(working))
    for term in snowball_tokenizer(keywords):
        scores += working["title_snowball"].array.score(term) * 10.0
        scores += working["description_snowball"].array.score(term) * 1.0
    top_idx = np.argsort(scores)[-top_k:][::-1]
    top_rows = working.iloc[top_idx].copy()
    top_rows.loc[:, "score"] = scores[top_idx]
    results = []
    for _, row in top_rows.iterrows():
        results.append({
            "id": row.get("doc_id", row.name),
            "title": row.get("title", ""),
            "description": row.get("description", ""),
            "score": row.get("score", 0.0),
            "category": row.get(column, ""),
        })
    return results

passage_fn = lambda row: f"{row.get('title','')}

{row.get('description','')}".strip()
embeddings, model = load_or_create_embeddings(corpus, passage_fn=passage_fn, model_name=DEFAULT_MODEL_NAME)
if model is None:
    model = load_model(DEFAULT_MODEL_NAME)

def e5_wands_prefiltered(query, category, top_k=10, column=CATEGORY_COL):
    if top_k > 100:
        raise ValueError("top_k must be <= 100")
    index = _category_index(corpus, column=column)
    if category not in index:
        return []
    indices = index[category]
    working_embeddings = embeddings[indices]
    working_corpus = corpus.iloc[indices]
    query_emb = np.asarray(model.encode(query))
    scores = working_embeddings @ query_emb
    top_idx = np.argsort(scores)[-top_k:][::-1]
    results = []
    for i in top_idx:
        row = working_corpus.iloc[i]
        results.append({
            "id": row.get("doc_id", row.name),
            "title": row.get("title", ""),
            "description": row.get("description", ""),
            "score": float(scores[i]),
            "category": row.get(column, ""),
        })
    return results

top_categories(5)

## Agent definitions and plan

We define pydantic response models and the select/scatter/gather steps.

In [ ]:
from pydantic import BaseModel, Field
from cheat_at_search.agent.openai_agent import OpenAIAgent

class SelectedCategories(BaseModel):
    categories: list[str] = Field(description="Selected categories to search within.")

class SearchResults(BaseModel):
    ranked_results: list[str] = Field(description="Top ranked search results (doc_ids).")

def run_select(query):
    tools = [bm25_wands, top_categories]
    system_prompt = (
        "You're trying to help find products in a furniture / home goods dataset. "
        "Use the provided tools to find the best categories to search within."
    )
    agent = OpenAIAgent(tools=tools, model="openai/gpt-5-mini", response_model=SelectedCategories, reasoning_level="medium")
    inputs = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Find the best categories to search within for the query: {query}"},
    ]
    resp = agent.loop(inputs=inputs)
    return resp.categories

def run_scatter(query, category):
    tools = [lambda q, top_k=10, agent_state=None: bm25_wands(q, [category], top_k=top_k), e5_wands_prefiltered]
    system_prompt = (
        "You're trying to help find products in a furniture / home goods dataset for a given search query. "
        "Use the provided tools to find the best results within the assigned category."
    )
    agent = OpenAIAgent(tools=tools, model="openai/gpt-5-mini", response_model=SearchResults, reasoning_level="medium")
    inputs = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"You're searching category: {category} for the user's query: {query}. Find the best results within this category."},
    ]
    resp = agent.loop(inputs=inputs)
    return resp.ranked_results

def _format_results(results_by_category):
    lines = ["Results by category:"]
    for category, items in results_by_category.items():
        lines.append(f"\n## {category}")
        if not items:
            lines.append("- (no results)")
            continue
        for item in items:
            lines.append(f"- id: {item['id']}")
            if item.get('title'):
                lines.append(f"  title: {item['title']}")
            if item.get('description'):
                lines.append(f"  description: {item['description']}")
            if item.get('category'):
                lines.append(f"  category: {item['category']}")
    return "\n".join(lines)

def run_gather(query, results_by_category):
    system_prompt = (
        "You're trying to rerank products from a furniture / home goods dataset for a given query. "
        "Given the candidate search results you've been provided, return the most relevant results."
    )
    agent = OpenAIAgent(tools=[], model="openai/gpt-5-mini", response_model=SearchResults, reasoning_level="medium")
    inputs = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": f"Rank the most relevant results to the users query: {query}. Rank these results: {_format_results(results_by_category)}"},
    ]
    resp = agent.loop(inputs=inputs)
    return resp.ranked_results

## Run a single scatter-gather query

We run select, scatter for each category, then gather the final ranked list.

In [ ]:
query = "salon chair"
categories = run_select(query)
categories

In [ ]:
results_by_category = {}
for category in categories:
    doc_ids = run_scatter(query, category)
    rows = []
    for doc_id in doc_ids:
        row = corpus[corpus["doc_id"] == int(doc_id)].iloc[0]
        rows.append({
            "id": str(doc_id),
            "title": row.get("title", ""),
            "description": row.get("description", ""),
            "category": row.get(CATEGORY_COL, ""),
        })
    results_by_category[category] = rows

final_ranked = run_gather(query, results_by_category)
final_ranked[:10]

## Evaluate on a few sampled queries

We wrap the scatter-gather logic in a SearchStrategy so we can call run_strategy and compute NDCG.

In [ ]:
from cheat_at_search.strategy import SearchStrategy
from cheat_at_search.search import run_strategy, ndcgs

class ScatterGatherStrategy(SearchStrategy):
    def __init__(self, corpus, workers=1):
        super().__init__(corpus, workers=workers)
        self.index = corpus
    def search(self, query, k=10):
        categories = run_select(query)
        results_by_category = {}
        for category in categories:
            doc_ids = run_scatter(query, category)
            rows = []
            for doc_id in doc_ids:
                row = corpus[corpus["doc_id"] == int(doc_id)].iloc[0]
                rows.append({
                    "id": str(doc_id),
                    "title": row.get("title", ""),
                    "description": row.get("description", ""),
                    "category": row.get(CATEGORY_COL, ""),
                })
            results_by_category[category] = rows
        ranked = run_gather(query, results_by_category)
        # Map doc_ids to corpus indices
        lookup = {str(doc_id): idx for idx, doc_id in corpus["doc_id"].items()}
        ilocs = [lookup[d] for d in ranked if d in lookup]
        scores = list(reversed(range(1, len(ilocs) + 1)))
        return ilocs, scores

strategy = ScatterGatherStrategy(corpus, workers=1)
graded = run_strategy(strategy, judgments, num_queries=3, seed=123, cache=False)
ndcgs(graded).mean()